<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

**Photostimulation Epoch** 

During the photostimulation epochs, single neurons were photostimulated to uncover causal connections between neurons in the field of view. The data from the photostimulation epochs before and after the BCI task can be used to assess how connectivity changes with learning. This tutorial will go through key experimental features of the photostimulation epoch, how to load and work with the data, and provide a few ideas for further analysis. 
        
**Recommended background resources:** 
* [Credit Assignment During Learning Project Page](https://www.allenneuraldynamics.org/projects/credit-assignment-during-learning) 
    
* [SWDB Data Book - Documentation and Code Tutorials](https://allenswdb.github.io/physiology/ophys/BCI/BCI-dataset.html)
    
* If this is your first time working with this dataset, please start with the `bci_demo` python notebook in the /code folder of this capsule. 


In [ ]:
# General imports 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Pynwb imports
from hdmf_zarr import NWBZarrIO

In [ ]:
# # pre-stored filepath to nwb file 
nwb_path = '/data/brain-computer-interface-v2/single-plane-ophys_754303_2025-01-27_20-01-31_processed_2025-08-07_06-00-10/single-plane-ophys_754303_2025-01-27_20-01-31_behavior_nwb'

In [ ]:
with NWBZarrIO(str(nwb_path), 'r') as io:
    nwbfile = io.read()
    print('Loaded NWB file from:', nwb_path)
    
nwbfile

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

**Stimulus table (2p optogenetic activation stimulus)** 
    
Information about the photostimulation trials can be found in the stimulus container under the name "PhotostimTrials". The data is stored as a DynamicTable which we will load as a pandas DataFrame for easier access. Each row represents a unique photostimulation trial. A full description of each column is below. 

| Column    | Description |
| -------- | ------- |
| start_time  | stimulus start (s)  |
| stop_time | stimulus end (s)   |
| start_frame |  stimulus start (frame)  |
| stop_frame   |  stimulus end (frame)   |
| tiff_file  | data source file name   |
| stimulus_name   | stimulus name   |
| laser_x    | x coordinate of stimulated neuron (pixels)  |
| laser_y   | y coordinate of stimulated neuron (pixels)  |
| power    | stimulus intensity (mW) |
| duration    | trial duration (s)  |
| stimulus_function    | stimulus template  |
| group_index   | number identifier for stimuluated neuron  |
| closest_roi    | index in dff that corresponds to the photostimulated neuron  |


In [ ]:
photostim = nwbfile.stimulus["PhotostimTrials"].to_dataframe()
photostim

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

**Photostimulation Epoch: Experimental Design** 

Each session contains 2 photostimulation epochs - one before and one after the BCI epoch. (Note that there is some variability in the epoch structure across sessions. Double check the epoch table before working with session data). The rest of the tutorial will focus on the first photostim epoch, but the experimental design structure applies to both epochs. 

In [ ]:
photostim.stimulus_name.unique()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

**Serial 2p photostimulation of single neurons** 

During each photostimulation trial, a single neuron in the FOV is activated using 2p photostimulation. As a sanity check, let's plot the neural activity response from a photostimulated neuron during the trial. We should see the neuron's neural activity increases (the fluorescence becomes brighter) when the laser turns on. 
    
Select a photostimulation trial to plot. The `closest_roi` value for that trial tells us the index in the dff array that corresponds with the photostimulated neuron. 
    
Get the neural activity data from the dff array and plot during the trial window +/- 1s. The `start_frame` and `stop_frame` columns in the photostim table give the trial frame indices to plot. For visualization purposes, we'll convert frames to time by dividing by the frame_rate. 

In [ ]:
# Get dff array (neurons x frames time-series array of neural activity) and frame rate from nwb file 
dff = nwbfile.processing["processed"].data_interfaces["dff"].roi_response_series["dff"].data
print('dff shape:',np.shape(dff))

frame_rate = nwbfile.imaging_planes["processed"].imaging_rate
print('Frame Rate:', frame_rate) 

In [ ]:
trial_of_interest = 8 # select trial 
time_ext = 1 # select bounds for time window (in sec) 

# Get dff and frame indices from photostim table 
cell = photostim.closest_roi[trial_of_interest] 
start_f = photostim.start_frame[trial_of_interest] 
stop_f = photostim.stop_frame[trial_of_interest] 

# Plot neural activity trace for photostimulated neuron during selected trial 
trace = dff[start_f - int(time_ext*frame_rate):stop_f + int(time_ext*frame_rate), cell] # plot trace during photostim period +/- 1s 
# Set up array for plotting time 
t = np.arange(start_f - int(time_ext*frame_rate), stop_f + int(time_ext*frame_rate))/frame_rate 

# Plot 
plt.figure(figsize=[15, 5])
plt.plot(t, trace, 'k') 
plt.axvspan(start_f/frame_rate, stop_f/frame_rate, color = 'r', alpha = 0.5)
plt.xlabel('Time(s)') 
plt.ylabel('ΔF/F')

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Over the entire photostimulation epoch, 50 different neurons were photostimulated in random order for several repeats. We can explore aspects of the `closest_roi` column in the photostim table to find the unique photostimulated neurons and the number of times the photostimulation was repeated. 

In [ ]:
photostim[['closest_roi']]

In [ ]:
# How many unique single neurons were photostimulated? 

photostim.closest_roi.nunique()

In [ ]:
# Count the number of trials (rows) for each roi 
photostim['closest_roi'].value_counts()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

**Plot the neural activity responses to repeated photostimulation** 

Let's look at the variability in a single neuron's response to repeated photostimulation trials. 
    
The following code will iterate through each photostimulation trial and append the selected neuron's neural activity during each trial +/- 2s (defined by the `time_ext` variable). We'll plot each response (black) and the mean response across trials (red). 

In [ ]:
trial_of_interest = 8 # select trial 
time_ext = 2 # select time window bounds (sec) 

# Get data from photostim table 
cell = photostim.closest_roi[trial_of_interest] 
# Get photostim trial frames for selected ROI 
start_f = photostim[photostim.closest_roi == cell].start_frame.tolist()
stop_f = photostim[photostim.closest_roi == cell].stop_frame.tolist()
# Build time array for plotting 
t = np.arange(start_f[0] - int(time_ext/2*frame_rate), start_f[0] + int(time_ext*frame_rate))/frame_rate 

# Build an array of trials x frames (each row is a photostim trial for the selected neuron) 
stim_dff = [] 
for idx in range(0, len(start_f)): 
    trace = dff[start_f[idx] - int(time_ext/2*frame_rate):start_f[idx] + int(time_ext*frame_rate), cell] # plot trace during photostim period
    stim_dff.append(trace.tolist()) # store as an array 

plt.figure(figsize=[15, 5])
for idx in range(0, len(start_f)): 
    plt.plot(t, stim_dff[idx], 'k', alpha = 0.1) 
plt.plot(t, np.nanmean(stim_dff, axis=0), 'r')
plt.axvspan(start_f[0]/frame_rate, stop_f[0]/frame_rate, color = 'r', alpha = 0.3)
# plt.xticks([8.02, 9.02, 10.02, 11.02], ['-1', '0', '1', '2'])
plt.xlabel('Time(s)') 
plt.ylabel('ΔF/F')

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

**Plot the photostimulation response across the FOV** 

Photostimulation was used to probe causal connections across the network. Let's look at responses in other neurons after each photostimulation trial to quickly visualize patterns of connectivity. 
    
Plot the neural activity of all the photostimulated neurons during the first 46 trials. The code below will iterate through each trial and plot the neural activity for the photostimulated neuron over the first 46 trials. Instead of a line graph, we use heatmap to better visualize excitatory and inhibitory responses. 

In [ ]:
events = photostim[photostim.stimulus_name == 'photostim'].iloc[:46] # Plot first 46 trials for simplicity 
stim_frames = events['start_frame'].values

# Get first and last frames for plotting 
start_f = max(0, stim_frames[0])
stop_f = min(len(dff), stim_frames[-1])

# Get all indices for photostimulated neurons 
target_cells = events.closest_roi[:] 

# Set up time array for plotting 
t = np.arange(start_f, stop_f) / frame_rate

# Extract traces for each targeted cell over the time window 
traces = np.array([dff[start_f:stop_f, cell] for cell in target_cells])

# Plot heatmap
plt.imshow(traces, aspect='auto', cmap='seismic', vmin=-10, vmax=10,
           extent=[t[0], t[-1], 1, len(traces)],interpolation = 'none')
plt.xlabel('Time (s)')
plt.ylabel('Trial')
plt.title('Photostimulated neuron traces')
plt.colorbar(label='ΔF/F')
plt.show()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

### **Important note about photostimulation data:**

**Light Artifact During Photostimulation** 
    
The neural responses observed **during** the photostimulation period have a light artifact (GCaMP is is excited by the optogenetic stimulation light). When working with data from the photostimulation period, remove the light artifact by ignoring the neural activity during the photostimulation trials and considering responses for a time window after the trial ends. 

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

**Summary and Follow-up Ideas** 

The responses within a photostimulation epoch can be used to understand the underlying structure of connectivity. Some questions to explore: 
    
* How does connection strength depend on distance? (see start-up code below for help) 
    
* Do correlated neurons tend to have stronger connections? 
    
* How connected is any given neuron in the FOV? What is the distribution of the number of causal connections between neurons?
    
* Do differences in the transgenic lines or injected viruses impact connectivity and photostimulation measurements? 
    
The responses across photostimulation epochs can be used to understand how the causal connections in the network change after learning. For example: 
 
* How does connectivity change across the photostimulation periods? Does it change with the BCI task? 

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F7FAFC; ">

**How does connection strength depend on distance? (Start-up code)** 
    
First, calculate the Euclidean Distance* between each ROI and the photostimulated neuron. As an approximation of connection strength, measure the change in neural activity to photostimulation for each ROI. Plot the change in neural activity with respect to distance from the photostimulated neuron. 

-------------------
**Where to find spatial locations for each neuron:**
    
Photostimulated neuron: the coordinate (in pixels) for the photostimulated neuron is contained in the `photostim` table, `laser_x` and `laser_y` columns. 
    
ROI: the `image_segmentation` table within the NWB file contains image masks (HxW array that represents the FOV with masked out regions where the ROI is spatially located). The center coordinate of the image masks can be used as the spatial coordinate for each ROI. (We've written a function `get_roi_centroids` to calculate the center coordinate (in pixels) from the image masks below).  

*umPerPix = 1.7755681818181817 # Microns per pixel (scale for spatial calculations)


In [ ]:
photostim[['laser_x', 'laser_y']]

In [ ]:
image_segmentation = nwbfile.processing["processed"].data_interfaces["image_segmentation"].plane_segmentations["roi_table"].to_dataframe()
image_segmentation.head()

In [ ]:
def get_roi_centroids(image_segmentation):
    """ 
    Calculate the center coordinate from each image mask
    """ 
    centroids = []
    for mask in image_segmentation['image_mask']:
        ys, xs = np.where(mask)
        x = np.mean(xs)
        y = np.mean(ys)
        centroids.append((x, y))
    return np.array(centroids)

In [ ]:
centroids = get_roi_centroids(image_segmentation)
centroids